# HeterogeneousMTGP

## 1. このモデルを使う場面

`HeterogeneousMTGP` は、**タスクごとに入力空間が異なる**ときに使うマルチタスクGPです。

例えば、

- タスク0（目的タスク）: 温度・圧力・流量を使う
- タスク1（補助タスク）: 温度・流量だけを使う

のように、タスク間で利用可能な特徴量が一致しない場合を扱えます。

このNotebookでは、タスク0を予測対象とし、タスク1のデータを転移学習的に利用します。

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import HeterogeneousMTGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. 合成データ

完全な特徴空間を `[x0, x1, x2]` とします。

- タスク0: `[x0, x1, x2]`
- タスク1: `[x0, x2]`

`feature_indices` は各タスクの列が、完全な特徴空間のどこに対応するかを表します。

In [ ]:
def task0_function(X):
    return (
        torch.sin(2 * torch.pi * X[:, 0])
        + 0.7 * (X[:, 1] - 0.5)
        + 0.5 * torch.cos(2 * torch.pi * X[:, 2])
    ).unsqueeze(-1)

def task1_function(X):
    # タスク0と共通する x0, x2 の構造を持つ補助タスク
    return (
        0.85 * torch.sin(2 * torch.pi * X[:, 0])
        + 0.45 * torch.cos(2 * torch.pi * X[:, 1])
        + 0.15
    ).unsqueeze(-1)

train_X0 = torch.rand(18, 3)
train_Y0 = task0_function(train_X0) + 0.04 * torch.randn(18, 1)

train_X1 = torch.rand(28, 2)
train_Y1 = task1_function(train_X1) + 0.04 * torch.randn(28, 1)

feature_indices = [
    [0, 1, 2],  # タスク0: x0, x1, x2
    [0, 2],     # タスク1: x0, x2
]

print(train_X0.shape, train_X1.shape)
print(train_Y0.shape, train_Y1.shape)

## 3. モデル構築

BoTorch / robotorchan の `HeterogeneousMTGP` では **タスク0が目的タスク**です。
`train_Xs` 自体には task feature を含めません。

In [ ]:
model = HeterogeneousMTGP(
    train_Xs=[train_X0, train_X1],
    train_Ys=[train_Y0, train_Y1],
    train_Yvars=None,
    feature_indices=feature_indices,
    full_feature_dim=3,
    use_saas_prior=False,
)

## 4. robotorchan のrawデータAPI

通常の `raw_train_X` ではなく、タスクごとのデータを `raw_train_Xs` / `raw_train_Ys` として保持します。

In [ ]:
print("supports_mll:", model.supports_mll)
print("raw_train_Xs:", [x.shape for x in model.raw_train_Xs])
print("raw_train_Ys:", [y.shape for y in model.raw_train_Ys])
print("raw_train_Yvars:", model.raw_train_Yvars)
print("raw_data keys:", model.raw_data.keys())

## 5. 学習

In [ ]:
mll = model.make_mll()
fit_gpytorch_mll(mll)
model.eval()

## 6. 目的タスクのposterior

posteriorはタスク0に対して計算します。
現行BoTorch APIでは、入力末尾に task feature `0` を付けます。

In [ ]:
x0 = torch.linspace(0.0, 1.0, 120)
test_X = torch.stack(
    [
        x0,
        torch.full_like(x0, 0.5),
        torch.full_like(x0, 0.5),
    ],
    dim=-1,
)
test_X_with_task = torch.cat(
    [test_X, torch.zeros(test_X.shape[0], 1)],
    dim=-1,
)

with torch.no_grad():
    posterior = model.posterior(test_X_with_task)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)

truth = task0_function(test_X).squeeze(-1)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x0, truth, label="真値")
plt.plot(x0, mean, label="HeterogeneousMTGP")
plt.fill_between(
    x0,
    mean - 1.96 * std,
    mean + 1.96 * std,
    alpha=0.2,
    label="95%区間",
)
plt.xlabel("x0（x1=x2=0.5で固定）")
plt.ylabel("y")
plt.legend()
plt.show()

## 7. 使い分け

### HeterogeneousMTGPが向くケース
- タスクごとに入力次元や利用可能な特徴量が違う
- 補助タスクの知識を目的タスクへ転移したい
- 異なる実験系・設備・シミュレータ間で共通特徴が一部だけ存在する

### 通常のMultiTaskGPが向くケース
- 全タスクが同じ特徴空間を持つ
- task featureを付けるだけで表現できる

### 注意
`feature_indices` は「各タスク内の列が完全特徴空間のどこに対応するか」を正しく指定する必要があります。
また、このモデルのposteriorは目的タスク（task 0）に限定されます。